<h1 style="font-size:40px;"> Extendiendo Spark </h1>

![](img/spark-hub.jpg)


Cada vez la comunidad de Spark es más grande y podemos hacer más cosas, vamos a ver algunos ejemplos de qué más podemos hacer con Spark:

Empezamos iniciando la sesión:

In [1]:
import os
from functools import reduce

import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

In [2]:
conf = (
    SparkConf()
    .setAppName("[ICAI] Extendiendo Spark")
    .set("spark.executor.memory", "8g")
    .set("spark.executor.cores", "5")
    .set("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.2")
)

In [3]:
spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()

:: loading settings :: url = jar:file:/sharenfs/spark-3.5.3-bin-hadoop3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jayuso/.ivy2/cache
The jars for the packages stored in: /home/jayuso/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e3efa343-baeb-48db-afa6-81b097c84d55;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;3.0.2 in central
	found org.mongodb#mongodb-driver-sync;4.0.5 in central
	found org.mongodb#bson;4.0.5 in central
	found org.mongodb#mongodb-driver-core;4.0.5 in central
:: resolution report :: resolve 757ms :: artifacts dl 8ms
	:: modules in use:
	org.mongodb#bson;4.0.5 from central in [default]
	org.mongodb#mongodb-driver-core;4.0.5 from central in [default]
	org.mongodb#mongodb-driver-sync;4.0.5 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.12;3.0.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts 

## UDF: *User Defined Functions*

Con las `UDF` podemos extender las funcionalidad de spark *DataFrame* igual que haciamos con los `RDD` y utilizar cualquier función de python. Veamos un ejemplo:

In [4]:
audiencias = spark.read.load("/datos/ejercicio_audis/audiencias_large.parquet").cache()

In [5]:
cuantiles = audiencias.select(
    F.expr(""" percentile_approx(segundos_visualizados,array(0,.25,.5,.75,1)) as cuantiles """)
).first()[0]

In [6]:
cuantiles

[1, 183, 751, 2826, 83042]

In [7]:
import bisect

In [8]:
def findInterval(x):
    return bisect.bisect(cuantiles, x)

La función `findInterval` nos indica en qué intervalo se encuentra el valor de `x` respecto a los cuantiles.

In [9]:
findInterval(1000)

3

In [10]:
findInterval(4500)

4

Con `F.udf` convertimos la función `findInterval` para trabajar con spark, tenemos que definir el tipo que va a devolver (en este caso `integer`):

In [11]:
findInterval_udf = F.udf(findInterval, T.IntegerType())

In [12]:
audiencias_cuantiles = audiencias.select(findInterval_udf("segundos_visualizados").alias("cuantil"))

In [13]:
audiencias_cuantiles.show(5)

+-------+
|cuantil|
+-------+
|      4|
|      3|
|      1|
|      1|
|      4|
+-------+
only showing top 5 rows



In [14]:
audiencias_cuantiles.describe().show()

[Stage 6:=================================================>         (5 + 1) / 6]

+-------+------------------+
|summary|           cuantil|
+-------+------------------+
|  count|          51191302|
|   mean|2.4999108246943984|
| stddev| 1.118119973747811|
|    min|                 1|
|    max|                 5|
+-------+------------------+



In [15]:
prueba1 = audiencias.select(F.lower(F.split("franja", "_")[1]).alias("nuevo"))

In [16]:
prueba2 = audiencias.select((F.udf(lambda x: (x.split("_")[1]).lower())("franja")).alias("nuevo"))

In [17]:
prueba1.show(5)

+---------+
|    nuevo|
+---------+
|   manana|
|primetime|
|madrugada|
|    tarde|
|    noche|
+---------+
only showing top 5 rows



In [18]:
prueba2.show(5)

+---------+
|    nuevo|
+---------+
|   manana|
|primetime|
|madrugada|
|    tarde|
|    noche|
+---------+
only showing top 5 rows



Las `UDF` son muy versátiles y nos abre un gran mundo de posibilidades pero son más lentas que usar las funciones de spark así que siempre que podamos usaremos estas últimas

## Pandas UDF

Las [Pandas UDF](https://databricks.com/blog/2017/10/30/introducing-vectorized-udfs-for-pyspark.html) fueron introducidas en Spark 2.3. Tienen la misma idea que las UDF pero con mayor *performance*:

![](./img/pandas_udf.png)

Estas funciones consiguen mayor velocidad gracias al proyecto [Apache Arrow](https://arrow.apache.org/):



![](img/apache_arrow.png)

In [19]:
from pyspark.sql.functions import pandas_udf

In [20]:
# x = audiencias.select("franja").limit(20).toPandas().franja

In [21]:
@pandas_udf(T.StringType())
def pandas_tratar(x):
    return x.str.split("_").str[0].str.lower()

In [22]:
prueba5 = audiencias.select(pandas_tratar("franja").alias("nuevo"))

In [23]:
prueba5.show(5)

[Stage 11:>                                                         (0 + 1) / 1]

+-----+
|nuevo|
+-----+
|finde|
|finde|
|finde|
|finde|
|entre|
+-----+
only showing top 5 rows



### Programación más compleja usando python

In [24]:
listas = spark.read.load("/datos/listas/listas.parquet")

In [25]:
items = spark.read.load("/datos/listas/item.parquet")

In [26]:
(
    items.select(F.explode("categorias").alias("categorias"))
    .groupBy("categorias")
    .count()
    .orderBy(F.desc("count"))
).show()

[Stage 14:=====>                                                   (1 + 9) / 10]

+----------+-----+
|categorias|count|
+----------+-----+
|        16| 2141|
|       128|  721|
|        80|  286|
|         0|  274|
|       112|  239|
|        96|   39|
|        64|   30|
|      NULL|    1|
|        48|    1|
+----------+-----+



**Paso 1**: Queremos obtener los 20 primeros items para la categoría 16 para cada usuario y tipo.

In [27]:
listas_16 = (
    listas.join(items.filter(F.array_contains("categorias", 16)), "id_item", "leftsemi")
    .withColumn(
        "rnk",
        F.row_number().over(Window.partitionBy("id_user", "tipo").orderBy(F.desc("rating"))),
    )
    .filter("rnk<=20")
    .drop("rating")
    .withColumn("categoria", F.lit(16))
)

In [28]:
listas_16.groupBy("id_user", "tipo").count().describe("count").show()

[Stage 20:======================================================> (44 + 1) / 45]

+-------+-------------------+
|summary|              count|
+-------+-------------------+
|  count|             161900|
|   mean|  19.98575046324892|
| stddev|0.39481101721079764|
|    min|                  1|
|    max|                 20|
+-------+-------------------+



**Paso 2**: Hacemos lo mismo para la categoría 112.

In [29]:
listas_112 = (
    listas.join(items.filter(F.array_contains("categorias", 112)), "id_item", "leftsemi")
    .withColumn(
        "rnk",
        F.row_number().over(Window.partitionBy("id_user", "tipo").orderBy(F.desc("rating"))),
    )
    .filter("rnk<=20")
    .drop("rating")
    .withColumn("categoria", F.lit(112))
)

In [30]:
listas_112.groupBy("id_user", "tipo").count().describe("count").show()

[Stage 31:=====================================================>  (48 + 2) / 50]

+-------+------------------+
|summary|             count|
+-------+------------------+
|  count|            160443|
|   mean|19.766272134028906|
| stddev|1.8875850461909092|
|    min|                 1|
|    max|                20|
+-------+------------------+



**Paso 3**: Unir las listas

In [31]:
listas_unidas = listas_16.union(listas_112)

In [32]:
listas_unidas.show()

[Stage 45:>                                                         (0 + 1) / 1]

+-------+-------+----+---+---------+
|id_item|id_user|tipo|rnk|categoria|
+-------+-------+----+---+---------+
|   11.0|   36.0|azul|  1|       16|
|  101.0|   36.0|azul|  2|       16|
|  158.0|   36.0|azul|  3|       16|
|    7.0|   36.0|azul|  4|       16|
|   43.0|   36.0|azul|  5|       16|
|  344.0|   36.0|azul|  6|       16|
|  274.0|   36.0|azul|  7|       16|
|  115.0|   36.0|azul|  8|       16|
|   83.0|   36.0|azul|  9|       16|
|   14.0|   36.0|azul| 10|       16|
|   15.0|   36.0|azul| 11|       16|
|   30.0|   36.0|azul| 12|       16|
|  258.0|   36.0|azul| 13|       16|
|    1.0|   36.0|azul| 14|       16|
|  250.0|   36.0|azul| 15|       16|
|  123.0|   36.0|azul| 16|       16|
| 1789.0|   36.0|azul| 17|       16|
|  239.0|   36.0|azul| 18|       16|
|  110.0|   36.0|azul| 19|       16|
|  131.0|   36.0|azul| 20|       16|
+-------+-------+----+---+---------+
only showing top 20 rows



**Paso 4**: Queremos construir un DF como `listas_unidas` para unas categorías dadas:

In [33]:
quiero = [16, 80, 112]

In [34]:
def generar_lista(x):
    return (
        listas.join(items.filter(F.array_contains("categorias", x)), "id_item", "leftsemi")
        .withColumn(
            "rnk",
            F.row_number().over(Window.partitionBy("id_user", "tipo").orderBy(F.desc("rating"))),
        )
        .filter("rnk<=20")
        .drop("rating")
        .withColumn("categoria", F.lit(x))
    )

Usamos el `map` de python para gener una lista de `DF` de spark:

In [35]:
map(generar_lista, quiero)

Con `reduce` unimos todos los `DFs`:

In [36]:
lista_final = reduce(lambda x, y: x.union(y), map(generar_lista, quiero)).cache()

In [37]:
lista_final

DataFrame[id_item: double, id_user: double, tipo: string, rnk: int, categoria: int]

In [38]:
lista_final.count()

9570210

In [39]:
lista_final.show()

+-------+-------+----+---+---------+
|id_item|id_user|tipo|rnk|categoria|
+-------+-------+----+---+---------+
|   11.0|   36.0|azul|  1|       16|
|  101.0|   36.0|azul|  2|       16|
|  158.0|   36.0|azul|  3|       16|
|    7.0|   36.0|azul|  4|       16|
|   43.0|   36.0|azul|  5|       16|
|  344.0|   36.0|azul|  6|       16|
|  274.0|   36.0|azul|  7|       16|
|  115.0|   36.0|azul|  8|       16|
|   83.0|   36.0|azul|  9|       16|
|   14.0|   36.0|azul| 10|       16|
|   15.0|   36.0|azul| 11|       16|
|   30.0|   36.0|azul| 12|       16|
|  258.0|   36.0|azul| 13|       16|
|    1.0|   36.0|azul| 14|       16|
|  250.0|   36.0|azul| 15|       16|
|  123.0|   36.0|azul| 16|       16|
| 1789.0|   36.0|azul| 17|       16|
|  239.0|   36.0|azul| 18|       16|
|  110.0|   36.0|azul| 19|       16|
|  131.0|   36.0|azul| 20|       16|
+-------+-------+----+---+---------+
only showing top 20 rows



In [40]:
lista_final.groupBy("id_user", "tipo", "categoria").count().describe("count").show()

[Stage 77:========================================================(13 + 1) / 13]

+-------+------------------+
|summary|             count|
+-------+------------------+
|  count|            482918|
|   mean|19.817463834439803|
| stddev|1.6688515269853816|
|    min|                 1|
|    max|                20|
+-------+------------------+



## Spark Packages

En la web https://spark-packages.org/, podemos encontrar multitud de paquetes para ampliar el uso de spark. Veamos un ejemplo para conectar a [MongoDB](https://www.mongodb.com/) una base de datos de tipo NoSQL ampliamente utilizada.

![](img/mongo.png)

![](img/spark-connector.png)

En la configuración de spark hemos cargado este paquete de la siguiente manera:


```python
.set("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.2")
```



#### Leer
De este modo podemos leer un `DF` desde MongoDB:

In [41]:
pelis = (
    spark.read.format("com.mongodb.spark.sql.DefaultSource")
    .option("uri", "mongodb://edge01.bigdata.alumnos.upcont.es/imdb.pelis")
    .load()
)

[Stage 84:>                                                         (0 + 1) / 1]24/11/03 22:13:52 ERROR TaskSetManager: Task 0 in stage 84.0 failed 4 times; aborting job


Py4JJavaError: An error occurred while calling o332.load.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 84.0 failed 4 times, most recent failure: Lost task 0.3 in stage 84.0 (TID 2817) (worker02.bigdata.alumnos.upcont.es executor 3): com.mongodb.MongoCommandException: Command failed with error 13 (Unauthorized): 'Command aggregate requires authentication' on server edge01.bigdata.alumnos.upcont.es:27017. The full response is {"ok": 0.0, "errmsg": "Command aggregate requires authentication", "code": 13, "codeName": "Unauthorized"}
	at com.mongodb.internal.connection.ProtocolHelper.getCommandFailureException(ProtocolHelper.java:175)
	at com.mongodb.internal.connection.InternalStreamConnection.receiveCommandMessageResponse(InternalStreamConnection.java:302)
	at com.mongodb.internal.connection.InternalStreamConnection.sendAndReceive(InternalStreamConnection.java:258)
	at com.mongodb.internal.connection.UsageTrackingInternalConnection.sendAndReceive(Us

In [ ]:
pelis.printSchema()

In [ ]:
pelis.orderBy(F.desc("ratingvalue")).limit(10).toPandas()

#### Escribir
Del mismo modo podemos escribir en el mongodb:

In [ ]:
catalogo = spark.read.json("/datos/catalogo.json")

In [ ]:
catalogo.show()

In [ ]:
(
    catalogo.write.format("com.mongodb.spark.sql.DefaultSource")
    .mode("overwrite")
    .option("uri", "mongodb://edge01.bigdata.alumnos.upcont.es")
    .option("database", os.environ.get("USER"))
    .option("collection", "catalogo")
    .save()
)

In [42]:
spark.stop()